In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.DataFrame([[8,8,1],[7,9,1],[6,10,0],[5,12,0]], columns=['cgpa','profile_score','placed'])
df

,cgpa,profile_score,placed
0,8,8,1
1,7,9,1
2,6,10,0
3,5,12,0


In [3]:
def innitialize_parameter(layer_dim):
    np.random.seed(19)
    parameters = {}
    L = len(layer_dim)

    for i in range(1,L): # starts with 1 because layer 0 is input layer does not have weight and biases
        parameters['w'+str(i)] = np.ones((layer_dim[i-1], layer_dim[i]))*0.1   # no of rowa , no of col, innitial val
        parameters['b'+str(i)] = np.zeros((layer_dim[i], 1))
    
    return parameters


In [4]:
# utility function

def sigmoid(Z):
    return 1/(1 + np.exp(-Z))

In [5]:
# layer_dims = [3, 5, 2]

def forward_propogation(X,parameters): # X = Input layer

    A = X
    L = len(parameters) // 2 # 4 w1,b1,w2,b2

    for i in range(1,L+1): # 1,2,3,4
        A_prev = A
        w = parameters['w'+str(i)]
        b = parameters['b'+str(i)]

        

        print('A'+str(i-1)+': \n', A_prev)
        print('w'+str(i)+': \n', w)
        print('b'+str(i)+': \n', b)
        print('--'*10)

        Z = np.dot(w.T,A_prev) + b   # performs summation operations
        A = sigmoid(Z)
        print('A'+str(i)+': \n', A)
        print('--'*20)
    
    return A,A_prev

In [9]:
# updating the 9 parameters
def update_parameters(parameters, y, y_hat, A1, X):
    
    # Update W2 and b2
    parameters['w2'][0][0] = parameters['w2'][0][0] + (0.0001 * (y - y_hat) * A1[0][0])
    parameters['w2'][1][0] = parameters['w2'][1][0] + (0.0001 * (y - y_hat) * A1[1][0])
    parameters['b2'][0][0] = parameters['b2'][0][0] + (0.0001 * (y - y_hat))

    # Update w1 (first neuron)
    parameters['w1'][0][0] = parameters['w1'][0][0] + (
        0.0001 * (y - y_hat) * parameters['w2'][0][0] * A1[0][0] * (1 - A1[0][0]) * X[0][0])
    
    parameters['w1'][0][1] = parameters['w1'][0][1] + (
        0.0001 * (y - y_hat) * parameters['w2'][0][0] * A1[0][0] * (1 - A1[0][0]) * X[1][0])
    
    parameters['b1'][0][0] = parameters['b1'][0][0] + (
        0.0001 * (y - y_hat) * parameters['w2'][0][0] * A1[0][0] * (1 - A1[0][0]))


    # Update w1 (second neuron)
    parameters['w1'][1][0] = parameters['w1'][1][0] + (
        0.0001 * (y - y_hat) * parameters['w2'][1][0] * A1[1][0] * (1 - A1[1][0]) * X[0][0])
    
    parameters['w1'][1][1] = parameters['w1'][1][1] + (
        0.0001 * (y - y_hat) * parameters['w2'][1][0] * A1[1][0] * (1 - A1[1][0]) * X[1][0])
    
    parameters['b1'][1][0] = parameters['b1'][1][0] + (
        0.0001 * (y - y_hat) * parameters['w2'][1][0] * A1[1][0] * (1 - A1[1][0]))

    return parameters

In [ ]:
epoches = 20
np.random.seed(19)
parameters = innitialize_parameter([2,2,1])
# print(parameters)
avg_loss = []

X = df[['cgpa','profile_score']].values
Y = df[['placed']].values

for i in range(epoches):

    total_loss = 0
    # print(f"{i+1} - iteration")
    # print('--'*20)
    for j in range(X.shape[0]):

        # randomly select any one of the 4 data-rows
        row_num = np.random.randint(0,X.shape[0])

        # define X and Y
        X_data = X[row_num].reshape(2,1)
        Y_data = Y[row_num][0]

        # forward propagation
        Y_hat,a1 = forward_propogation(X_data,parameters)
        Y_hat = Y_hat[0][0]

        # loss calculation
        loss = (Y_data-Y_hat)**2
        total_loss += loss
        # print(loss)

        # update weights and bias
        update_parameters(parameters,Y_data,Y_hat,a1,X_data)
        # print(parameters)
        # print('--'*10+'\n')

    # calculate the average loss
    avg_loss.append(total_loss/X.shape[0])

In [11]:
parameters

{'w1': array([[0.10005911, 0.09993559],
        [0.10005915, 0.09993555]]),
 'b1': array([[6.40759683e-07],
        [6.41903274e-07]]),
 'w2': array([[0.0999563 ],
        [0.09995628]]),
 'b2': array([[-3.31294291e-05]])}

In [12]:
avg_loss

[np.float64(0.25173748211297864),
 np.float64(0.25191849380074466),
 np.float64(0.2726409518313735),
 np.float64(0.2517299182115338),
 np.float64(0.27259362951574584),
 np.float64(0.2729146549188327),
 np.float64(0.25207973965412883),
 np.float64(0.2312608864054404),
 np.float64(0.231058629019749),
 np.float64(0.2520876529950116),
 np.float64(0.21032996448957209),
 np.float64(0.23099577690595918),
 np.float64(0.23097475293696026),
 np.float64(0.23113590087374633),
 np.float64(0.23111516945890587),
 np.float64(0.27310358149295855),
 np.float64(0.252107946275842),
 np.float64(0.25174291222971706),
 np.float64(0.27286713566888254),
 np.float64(0.2520997975631125)]

## converging loss using keras

In [13]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,Input

In [14]:
model = Sequential([
    Input(shape=(2,)), # shape takes tupil as input can not just type in 2 
    Dense(2, activation='sigmoid'),
    Dense(1, activation='sigmoid')
])

E0000 00:00:1777558647.249273    5802 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [15]:
model.get_weights()

[array([[ 0.4963423 ,  1.126763  ],
        [ 0.7672725 , -0.29171872]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[-0.13356805],
        [-0.09269977]], dtype=float32),
 array([0.], dtype=float32)]

In [19]:
new_weights = [np.array([[ 0.1 , 0.1],
                         [ 0.1 , 0.1]], dtype=np.float32), np.array([0., 0.], dtype=np.float32), # 4 weights and 2 bias
              np.array([[0.1],
                        [0.1]], dtype=np.float32), np.array([0.], dtype=np.float32)] # 2 weights and 1 bias

In [20]:
model.set_weights(new_weights)

In [21]:
model.get_weights()

[array([[0.1, 0.1],
        [0.1, 0.1]], dtype=float32),
 array([0., 0.], dtype=float32),
 array([[0.1],
        [0.1]], dtype=float32),
 array([0.], dtype=float32)]

In [22]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 2)              │             6 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9 (36.00 B)

 Trainable params: 9 (36.00 B)

 Non-trainable params: 0 (0.00 B)

In [23]:
optimizer = keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss='binary_crossentropy', optimizer=optimizer)

In [24]:
history = model.fit(df.iloc[:, 0:-1], df.iloc[:, -1], epochs=75, verbose=1, batch_size=1)

Epoch 1/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6981  
Epoch 2/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6971 
Epoch 3/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6968 
Epoch 4/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6968 
Epoch 5/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6970 
Epoch 6/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6967 
Epoch 7/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6965 
Epoch 8/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6968 
Epoch 9/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6964 
Epoch 10/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6966 
Epoch 11/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6963 
Epoch 12/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6963 
Epoch 13/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6961 
Epoch 14/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6962 
Epoch 15/75
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6961 
Epoch 16/75
4/4 ━━━━━━━━━━━━━━━━━